# Session 08: Practice Quiz

**Notebook 2 of 3: Self-check** · about 35 minutes

For each question:

1. Write down your answer **before** opening "Show answer", or before running the code cell.
2. Run the cell where there is one.
3. Compare, and make sure you understand the *reasoning* in the answer, not only the result.

This quiz is formative: it's for you, and it isn't graded. As in the concepts notebook, cells can `await` directly because Jupyter runs an event loop.

In [ ]:
import asyncio
import time

import httpx

print("ready")

## Q1 · Predict: the order of the output

Write down the six lines this prints, in order.

In [ ]:
async def step(name: str, delay: float) -> str:
    print(f"start {name}")
    await asyncio.sleep(delay)
    print(f"end {name}")
    return name


results = await asyncio.gather(step("A", 0.2), step("B", 0.1), step("C", 0.0))
print(results)

<details>
<summary>Show answer</summary>

```text
start A
start B
start C
end C
end B
end A
['A', 'B', 'C']
```

- `gather` starts all three. Each runs until its first `await`, in argument order, so the three `start` lines come first.
- They then finish in order of their delays: C (0 s), B, A.
- The **results** come back in **argument** order, not completion order. That's what makes `zip(teams, results)` safe.

</details>

## Q2 · Predict: how long?

Predict the elapsed time for each version, then run it.

In [ ]:
async def wait(seconds: float) -> None:
    await asyncio.sleep(seconds)


started = time.perf_counter()
await wait(0.2)
await wait(0.2)
print(f"a: {time.perf_counter() - started:.1f}s")

started = time.perf_counter()
first = asyncio.create_task(wait(0.2))
second = asyncio.create_task(wait(0.2))
await first
await second
print(f"b: {time.perf_counter() - started:.1f}s")

started = time.perf_counter()
first = wait(0.2)
second = wait(0.2)
await first
await second
print(f"c: {time.perf_counter() - started:.1f}s")

<details>
<summary>Show answer</summary>

- **a: 0.4 s.** Each `await` runs the coroutine to completion before the next line.
- **b: 0.2 s.** `create_task` **schedules** both immediately; they run concurrently while you await the first.
- **c: 0.4 s.** Calling `wait(0.2)` only creates a coroutine object. Nothing runs until it's awaited, and each `await` then runs it to completion, one after the other.

Case c is the trap for JavaScript and C# developers: there, `const p = wait(200)` already starts the work. In Python, a coroutine object is lazy; a **Task** is what starts work early.

</details>

## Q3 · Debug: the dashboard endpoint that slows everything down

After a deploy, the whole incident service became slow, including `/health`, which does nothing. This endpoint was added in the deploy. Find the problem, and give two different fixes.

```python
@router.get("/dashboard")
async def dashboard(service: ServiceDep) -> Dashboard:
    response = httpx.get(f"{INVENTORY_URL}/assets", params={"team": "network"}, timeout=5)
    ...
```

<details>
<summary>Show answer</summary>

`httpx.get` is the **synchronous** API. Called inside an `async def` endpoint, it blocks the event loop for the whole round trip, and every other request to the service, including `/health`, queues behind it.

Fixes:

1. **Use the async client**, shared across requests: `response = await inventory.get("/assets", params=...)`, where `inventory` is an `httpx.AsyncClient` created at startup and injected with `Depends`. This is the workbook's design.
2. **Make the endpoint a plain `def`**. FastAPI then runs it in a thread pool, and the blocking call only blocks that thread. Correct, but it uses a thread per in-flight request and can't easily make concurrent calls.

(`await asyncio.to_thread(httpx.get, ...)` also works, but if an async client exists, use it.)

A single-request test wouldn't show this bug. It appears only under concurrent load, which is why the concepts notebook's FastAPI demo sends four requests at once.

</details>

## Q4 · Predict: one failure in `gather`

What does this print? And how many times does `"finished"` appear?

In [ ]:
async def job(name: str) -> str:
    await asyncio.sleep(0.1 if name != "bad" else 0.05)
    if name == "bad":
        raise ValueError(f"{name} failed")
    print(f"  {name} finished")
    return name


try:
    print(await asyncio.gather(job("a"), job("bad"), job("c")))
except ValueError as error:
    print("caught:", error)
await asyncio.sleep(0.2)  # give any remaining work time to show itself

<details>
<summary>Show answer</summary>

```text
caught: bad failed
  a finished
  c finished
```

- `gather` raises the **first** exception as soon as it happens (at 0.05 s), so the `try` block ends without any results.
- The other two coroutines are **not cancelled**: they keep running and finish later, with nobody using their results. In a real service, that's wasted work, and possibly side effects (a write, a note) you thought didn't happen.

`asyncio.TaskGroup` would cancel `a` and `c` when `bad` failed. `gather(..., return_exceptions=True)` would return `['a', ValueError('bad failed'), 'c']`.

</details>

## Q5 · Implement: a deadline per call

Complete `fetch_with_deadline` so that it returns the coroutine's result, or the string `"timed out"` if it takes longer than `seconds`. Use `asyncio.timeout`.

In [ ]:
async def slow_answer(delay: float) -> str:
    await asyncio.sleep(delay)
    return f"answered after {delay}s"


async def fetch_with_deadline(delay: float, seconds: float) -> str:
    # Your code here
    return await slow_answer(delay)


started = time.perf_counter()
print(await asyncio.gather(fetch_with_deadline(0.1, 0.5), fetch_with_deadline(2.0, 0.3)))
print(f"total: {time.perf_counter() - started:.1f}s")
# Expected: ['answered after 0.1s', 'timed out'], total about 0.3s

<details>
<summary>Show answer</summary>

```python
async def fetch_with_deadline(delay: float, seconds: float) -> str:
    try:
        async with asyncio.timeout(seconds):
            return await slow_answer(delay)
    except TimeoutError:
        return "timed out"
```

- The deadline **cancels** the slow call at 0.3 s; it doesn't just stop waiting for it.
- Each call has its **own** deadline, so the fast one isn't affected. That's the workbook's per-team deadline.
- Catch `TimeoutError` **outside** the `async with` block: the block converts the internal `CancelledError` into `TimeoutError` as it exits. (`asyncio.wait_for(coro, seconds)` is an older equivalent.)

</details>

## Q6 · Engineering judgment: will async help?

For each task, would rewriting it with `asyncio` make it substantially faster? Why?

1. Calling an LLM API for 200 support tickets, one request each, about 3 s per request.
2. Resizing 10,000 images with a pure-Python image library.
3. Reading 50 small JSON files from a local SSD.
4. A FastAPI endpoint that runs one database query and returns the result.
5. Checking the status pages of 300 internal services.

<details>
<summary>Show answer</summary>

1. **Yes, dramatically.** Almost all the time is waiting on the network. With, say, 20 in flight at a time (respecting the provider's rate limit), 200 × 3 s = 10 minutes becomes about 30 s.
2. **No.** It's CPU-bound; there's nothing to overlap, and the GIL stops threads from helping pure-Python code. Use processes or a native library.
3. **Not much.** Local file reads are fast, and asyncio has no native async file I/O (libraries use threads). Plain synchronous code is simpler, and probably just as fast.
4. **Not for that request**: it still waits for its one query. Async helps the **service** handle many concurrent requests efficiently, and only if the database driver is async; otherwise use a `def` endpoint.
5. **Yes.** 300 independent network waits are the textbook case. Add a `Semaphore` and a timeout per call, and handle failures per service.

The question to ask is always: **where is the time spent, waiting or computing?**

</details>

## Q7 · Debug: the polite crawler that isn't

This is meant to fetch at most **3** pages at a time. The peak shows otherwise. Why, and what's the fix?

In [ ]:
in_flight = 0
peak = 0


async def fake_page(request: httpx.Request) -> httpx.Response:
    global in_flight, peak
    in_flight += 1
    peak = max(peak, in_flight)
    await asyncio.sleep(0.05)
    in_flight -= 1
    return httpx.Response(200, text="ok")


async def fetch(client: httpx.AsyncClient, url: str) -> int:
    limit = asyncio.Semaphore(3)
    async with limit:
        response = await client.get(url)
        return response.status_code


async with httpx.AsyncClient(transport=httpx.MockTransport(fake_page)) as client:
    await asyncio.gather(*(fetch(client, f"http://site.test/page/{n}") for n in range(12)))
print("peak in flight:", peak)

<details>
<summary>Show answer</summary>

Each call to `fetch` creates its **own** semaphore, so each one limits only itself, and all 12 requests run at once. A limit must be **shared** by everything it limits:

```python
limit = asyncio.Semaphore(3)          # created once, outside


async def fetch(client: httpx.AsyncClient, url: str) -> int:
    async with limit:
        response = await client.get(url)
        return response.status_code
```

Now the peak is 3. The same mistake happens with clients (a new `AsyncClient` per call has its own connection pool and limits), and with locks.

</details>

## Q8 · Predict: does `except Exception` stop the timeout?

```python
async def careful_fetch() -> str:
    try:
        await asyncio.sleep(2)
        return "data"
    except Exception:
        return "fallback"

async with asyncio.timeout(0.1):
    result = await careful_fetch()
```

Does `careful_fetch` return `"fallback"`, or does the timeout still raise?

In [ ]:
async def careful_fetch() -> str:
    try:
        await asyncio.sleep(2)
        return "data"
    except Exception:
        return "fallback"


try:
    async with asyncio.timeout(0.1):
        result = await careful_fetch()
    print("result:", result)
except TimeoutError:
    print("TimeoutError: the timeout still worked")

<details>
<summary>Show answer</summary>

**The timeout still works.** The timeout cancels the coroutine by raising `asyncio.CancelledError` at its `await`, and `CancelledError` derives from `BaseException`, not `Exception`, so `except Exception` doesn't catch it.

That's deliberate: cancellation must pass through ordinary error handling. If you do write `except BaseException:` or `except asyncio.CancelledError:` (for example to log), **re-raise** it, or timeouts and shutdowns stop working.

</details>

## Q9 · Compare: Promises, Tasks and coroutines

A colleague from a TypeScript background writes:

```python
async def notify(team: str) -> None:
    await asyncio.sleep(0.1)
    print(f"notified {team}")

async def close_ticket(ticket_id: str) -> None:
    notify("network")            # fire and forget, like in TypeScript
    print(f"closed {ticket_id}")
```

1. What happens in Python, and what would happen in TypeScript?
2. How do you really "fire and forget" in Python, and what's the risk?

<details>
<summary>Show answer</summary>

1. In **Python**, `notify("network")` creates a coroutine object and throws it away. The notification is **never sent**, and Python warns "coroutine 'notify' was never awaited". In **TypeScript**, calling an async function starts it immediately, so the notification would be sent (with any error unobserved).
2. To start work without waiting for it: `task = asyncio.create_task(notify("network"))`. The risks:
   - the event loop keeps only a **weak** reference to tasks, so you must keep a reference (for example in a set) or the task can be garbage-collected before it finishes;
   - nobody awaits it, so its exceptions are only logged when it's collected, easy to miss;
   - in a web service, the task can outlive the request, and is cancelled when the server shuts down.

   Usually, it's better to **await** the notification, run it with the other work in a `TaskGroup`, or hand it to a proper background-job system. FastAPI also has `BackgroundTasks` for small after-the-response jobs.

</details>

## Q10 · Engineering judgment: all or nothing?

For each operation, should one failing call fail the whole operation (`TaskGroup`), or should you return partial results with the failures marked? Justify each.

1. A dashboard showing assets for four teams.
2. Booking a trip: a flight, a hotel and a car, which must all succeed.
3. A RAG query that searches three document collections before asking the LLM.
4. Sending an incident alert to Slack, email and SMS.

<details>
<summary>Show answer</summary>

1. **Partial.** Three teams' data is still useful; mark the fourth as unavailable. (The workbook.)
2. **All or nothing**, and more: if one booking fails, the successful ones must be **undone**. `TaskGroup` cancels the calls still in flight, but completed calls need compensating actions. Concurrency doesn't remove the need for transactional thinking.
3. **Usually partial, with care.** Answering from two collections is often better than failing, but the answer might be missing the key document. Record which collections failed, and consider saying so in the response. A reasonable team could also choose "fail" if completeness matters more than availability.
4. **Partial**, and keep trying the others: the point of several channels is redundancy. Report which channels failed, and retry them.

There's no universal rule. The decision follows from **what a partial answer means to the user**, and that's a product decision as much as a technical one.

</details>

---

**Done?** Go back over any question you got wrong; each points to a part of `01-concepts.ipynb`. If you haven't yet, work through `03-async-debugging-lab.ipynb`. Then finish **Project 02** with the workbook.